In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from analysis.autocorrelation import autocorrelation
from analysis.calculations import lattice_spacing_su2, lattice_spacing_su3


In [ ]:
# ---------------------------------------------------------------------------
# Inverse scale setting: a [fm] --> beta
# ---------------------------------------------------------------------------

def beta_from_a_su2(a_fm: float) -> float:
    """Analytically invert lattice_spacing_su2.

    From: ln(t0/a^2) = 1.285 + 6.409*x - 0.7411*x^2,  x = beta - 2.6
    Quadratic formula for x, pick root in physical regime.
    """
    t0_phys = 0.01133  # fm^2
    rhs = np.log(t0_phys / a_fm**2)
    A, B, C = -0.7411, 6.409, 1.285 - rhs
    disc = B**2 - 4*A*C
    if disc < 0:
        raise ValueError(f"No real solution for a={a_fm} fm")
    x1 = (-B + np.sqrt(disc)) / (2*A)
    x2 = (-B - np.sqrt(disc)) / (2*A)
    x = x1 if abs(x1) < abs(x2) else x2
    return x + 2.6


def beta_from_a_su3(a_fm: float) -> float:
    """Invert lattice_spacing_su3 via numpy.roots on the cubic.

    From: ln(a/r0) = -1.6804 - 1.7331*x + 0.7849*x^2 - 0.4428*x^3,  x = beta - 6.0
    """
    r0 = 0.5  # fm
    rhs = np.log(a_fm / r0)
    coeffs = [-0.4428, 0.7849, -1.7331, -1.6804 - rhs]
    roots = np.roots(coeffs)
    real_roots = roots[np.isreal(roots)].real
    if len(real_roots) == 0:
        raise ValueError(f"No real solution for a={a_fm} fm")
    x = real_roots[np.argmin(np.abs(real_roots))]
    return float(x + 6.0)


# round-trip sanity check
_a = 0.1
assert abs(lattice_spacing_su2(beta_from_a_su2(_a)) - _a) < 1e-10
assert abs(lattice_spacing_su3(beta_from_a_su3(_a)) - _a) < 1e-10
print("Round-trip checks passed.")


In [ ]:
# ---------------------------------------------------------------------------
# Volume-preserving lattice adjustment
# ---------------------------------------------------------------------------

def adjust_lattice(T_ref: int, L_ref: int, a_ref_fm: float, a_new_fm: float,
                   open_bc: bool = False, n_exclude: int = 2) -> dict:
    """Compute (T_new, L_new) keeping physical volume constant.

    open_bc: n_exclude slices discarded per temporal boundary,
             physical temporal extent = (T - 2*n_exclude) * a.
    """
    ratio = a_ref_fm / a_new_fm
    L_new = round(L_ref * ratio)

    if open_bc:
        T_eff_ref = T_ref - 2 * n_exclude
        T_new = round(T_eff_ref * ratio) + 2 * n_exclude
        T_eff_new = T_new - 2 * n_exclude
    else:
        T_new = round(T_ref * ratio)
        T_eff_new = T_new

    V_fm4 = T_eff_new * L_new**3 * a_new_fm**4
    return {"T_new": T_new, "L_new": L_new, "T_eff_new": T_eff_new, "V_fm4": V_fm4}


In [ ]:
# ---------------------------------------------------------------------------
# Scan: beta=2.5 upward, reference T=16^3 x 16, open BC
# ---------------------------------------------------------------------------

T_ref, L_ref = 16, 16
beta_ref  = 2.5
a_ref_fm  = lattice_spacing_su2(beta_ref)
open_bc   = True
n_exclude = 2   # must match exclude_boundary_slices in your input file

beta_values = np.arange(2.5, 3.05, 0.05)

print(f"Reference: beta={beta_ref}, T={T_ref}, L={L_ref}, a={a_ref_fm:.4f} fm, open_bc={open_bc}")
print()
print(f"{'beta':>8} {'a [fm]':>10} {'T_new':>7} {'L_new':>7} {'T_eff':>7} {'V [fm^4]':>12}")
print("-" * 55)

for beta in beta_values:
    a = lattice_spacing_su2(beta)
    adj = adjust_lattice(T_ref, L_ref, a_ref_fm, a, open_bc=open_bc, n_exclude=n_exclude)
    print(f"{beta:8.2f} {a:10.4f} {adj['T_new']:7d} {adj['L_new']:7d} "
          f"{adj['T_eff_new']:7d} {adj['V_fm4']:12.6f}")


In [ ]:
# ---------------------------------------------------------------------------
# Scan: beta=6.0 upward, reference T=16^3 x 16, open BC  [SU(3)]
# ---------------------------------------------------------------------------

T_ref, L_ref = 16, 16
beta_ref  = 6.0
a_ref_fm  = lattice_spacing_su3(beta_ref)
open_bc   = True
n_exclude = 2   # must match exclude_boundary_slices in your input file

beta_values = np.arange(6.0, 6.55, 0.05)

print(f"Reference: beta={beta_ref}, T={T_ref}, L={L_ref}, a={a_ref_fm:.4f} fm, open_bc={open_bc}")
print()
print(f"{'beta':>8} {'a [fm]':>10} {'T_new':>7} {'L_new':>7} {'T_eff':>7} {'V [fm^4]':>12}")
print("-" * 55)

for beta in beta_values:
    a = lattice_spacing_su3(beta)
    adj = adjust_lattice(T_ref, L_ref, a_ref_fm, a, open_bc=open_bc, n_exclude=n_exclude)
    print(f"{beta:8.2f} {a:10.4f} {adj['T_new']:7d} {adj['L_new']:7d} "
          f"{adj['T_eff_new']:7d} {adj['V_fm4']:12.6f}")